In [5]:
import torch 
import pickle
from pathlib import Path
import yaml
from model import GPT, GPTConfig


# load reward model for evaluating nanoGPT responses
base_path = Path.cwd()

meta_data = base_path/ "meta.pkl"
meta_data = pickle.load(open(meta_data, "rb"))
with open('config/config_reward.yaml') as f:
    conf = yaml.load(f, Loader=yaml.FullLoader)
    # nested dictionary structure
    config = {}               
    for k, v in conf.items():
        for k2, v2 in v.items():
            config[k2] = v2

vocab_size = meta_data['vocab_size']
model_args = dict(n_layer=config['n_layer'], n_head=config['n_head'], n_embd=config['n_embd'], block_size=config['block_size'],
                    bias=config['bias'], vocab_size=vocab_size, dropout=config['dropout'], ) # start with model_args from command line
reward_model = GPT(GPTConfig(**model_args))
reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")
reward_model.load_state_dict(reward_model_dict)
reward_model = reward_model.cuda()
reward_model.eval()

#decoder
def evaluate_response(prompt):
    encoded = [ meta_data['stoi'][ch]  for ch in prompt]
    if len(encoded) == 0:
        return 0.0
    inputs = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)  # Add batch dimension
    device = next(reward_model.parameters()).device
    inputs = inputs.to(device)
    with torch.no_grad():
        outputs = reward_model(inputs)
    reward_score = outputs.item()
    return reward_score

number of parameters: 3.61M


/tmp/ipykernel_2252782/211508470.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  reward_model_dict = torch.load(base_path/"reward_model.pth", map_location="cpu")


In [6]:
shakspeare_sample  =  """
Men yet he shall show me that hath not held to see.

ROMEO:
Farewell; sweet love, and get thee hence; for I must not;
For I am sorry, come not hither at my last.

FRIAR LAURENCE:
My lord, so hate the ground is ashame.

FRIAR LAURENCE:
So long as I love the dangerous tongue.

ROMEO:
O Rosaline, make me not: I love the duke
From Rosaline, till I prove a honour of your
Love of his own design. O Romeo, Romeo!

ROMEO:
What say you?

FRIAR JOHN:
No, by and by?

FRIAR LAURENCE:
Blessed of foot, how our
"""


sample = shakspeare_sample.split('\n')

In [7]:
reward_sample_scores = [ evaluate_response(s) for s in sample]

In [8]:
for sentence,score in zip(sample, reward_sample_scores):
    print(f"{sentence} = {score}")

 = 0.0
Men yet he shall show me that hath not held to see. = 1.2402280569076538
 = 0.0
ROMEO: = 1.8862576484680176
Farewell; sweet love, and get thee hence; for I must not; = 1.2740410566329956
For I am sorry, come not hither at my last. = 1.2122976779937744
 = 0.0
FRIAR LAURENCE: = 1.8920409679412842
My lord, so hate the ground is ashame. = 1.2227786779403687
 = 0.0
FRIAR LAURENCE: = 1.8920409679412842
So long as I love the dangerous tongue. = 1.3430900573730469
 = 0.0
ROMEO: = 1.8862576484680176
O Rosaline, make me not: I love the duke = 1.7514971494674683
From Rosaline, till I prove a honour of your = 1.209750771522522
Love of his own design. O Romeo, Romeo! = 1.089758038520813
 = 0.0
ROMEO: = 1.8862576484680176
What say you? = 1.8812012672424316
 = 0.0
FRIAR JOHN: = 1.892436146736145
No, by and by? = 1.8172287940979004
 = 0.0
FRIAR LAURENCE: = 1.8920409679412842
Blessed of foot, how our = 1.1507611274719238
 = 0.0
